# Activity 9 — Convert classified pixels into coverage statistics

**Learning objective:** Turn the prediction raster into class-level area estimates.

The prediction values (for example 1–9) are **class IDs** from the `randomforest` field.  
They are not NDVI values.

In [ ]:
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Read the output raster

In [ ]:
RASTER_PATH = "Results/Efate_Invasive_Prediction.tif"

with rasterio.open(RASTER_PATH) as src:
    data = src.read(1)
    pixel_width, pixel_height = src.res
    raster_crs = src.crs
    nodata = src.nodata

pixel_area_m2 = abs(pixel_width * pixel_height)
pixel_area_ha = pixel_area_m2 / 10000

print("CRS:", raster_crs)
print("Resolution:", (pixel_width, pixel_height))
print("Area per pixel:", pixel_area_ha, "ha")

## 2. Remove NoData and count each predicted class

In [ ]:
valid_data = data[np.isfinite(data)]

values, counts = np.unique(
    valid_data,
    return_counts=True
)

coverage = pd.DataFrame({
    "Class_ID": values.astype(int),
    "Pixel_Count": counts.astype(int),
    "Area_ha": counts * pixel_area_ha
})

total_area_ha = coverage["Area_ha"].sum()

coverage["Percent_of_Mapped_Area"] = (
    coverage["Area_ha"] / total_area_ha * 100
)

coverage

## 3. Add the class names

**Instructor step:** replace the names below with the confirmed class legend from the training data.

Do not guess the meaning of classes 1–9.

In [ ]:
CLASS_NAMES = {
    # 1: "Class name",
    # 2: "Class name",
    # 3: "Class name",
    # ...
}

coverage["Class_Name"] = coverage["Class_ID"].map(CLASS_NAMES)

coverage

In [ ]:
# Save the table for reporting.
coverage.to_csv(
    "Results/Efate_Class_Coverage_Statistics.csv",
    index=False
)

print("Coverage table saved.")

### Discussion

Once the class legend is confirmed, this table can be filtered to show only the invasive species.

This is the point where the workflow changes from **a map** to **management information**.